In [4]:
import pandas as pd
import numpy as np

In [5]:
# 1. Данные о простоях
df_downtime_repair_filter = pd.DataFrame({
    'Машина': ['A001', 'A002', 'B40'],
    'Оборудование': ['E001', 'E002', 'E960'],
    'Дата': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-10']),
    'Простой_часы': [5, 3, 10],
    'Простой_id': [522, 326, 101]
})

In [6]:
df_downtime_repair_filter

,Машина,Оборудование,Дата,Простой_часы,Простой_id
0,A001,E001,2024-01-01,5,522
1,A002,E002,2024-01-02,3,326
2,B40,E960,2024-01-10,10,101


In [7]:
# 2. Данные о работах (НЕСКОЛЬКО работ на один простой)
df_materials = pd.DataFrame({
    'Заказ на ремонт.Транспортное средство (АПК)': [
        'A001', 'A001', 'A001',  # 3 работы на A001
        'E002', 'E002',         # 2 работы на E002
        'A003'                  # лишняя запись
    ],
    'Дата': pd.to_datetime([
        '2024-01-01', '2024-01-01', '2024-01-01',
        '2024-01-02', '2024-01-02',
        '2024-01-03'
    ]),
    'Ответственное подразделение': [
        'Отдел А', 'Отдел Б', 'Отдел В',
        'Отдел Г', 'Отдел Д',
        'Отдел Е'
    ],
    'Номер_заказа': [
        'R-001', 'R-002', 'R-003',
        'R-004', 'R-005',
        'R-006'
    ],
    'Стоимость': [
        1000, 2000, 3000,
        4000, 5000,
        6000
    ]
})
df_materials

,Заказ на ремонт.Транспортное средство (АПК),Дата,Ответственное подразделение,Номер_заказа,Стоимость
0,A001,2024-01-01,Отдел А,R-001,1000
1,A001,2024-01-01,Отдел Б,R-002,2000
2,A001,2024-01-01,Отдел В,R-003,3000
3,E002,2024-01-02,Отдел Г,R-004,4000
4,E002,2024-01-02,Отдел Д,R-005,5000
5,A003,2024-01-03,Отдел Е,R-006,6000


In [8]:
# Делаем merge по машине
df_result_expanded = df_downtime_repair_filter.merge(
    df_materials,
    how='left',
    left_on=['Машина', 'Дата'],
    right_on=['Заказ на ремонт.Транспортное средство (АПК)', 'Дата']
)
df_result_expanded

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0
3,A002,E002,2024-01-02,3,326,NaN,NaN,NaN,NaN
4,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN


In [9]:
df_result_expanded.loc[df_result_expanded['Ответственное подразделение'].notna(), 'источник'] = 'по_машине'

In [10]:
df_result_expanded

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине
3,A002,E002,2024-01-02,3,326,NaN,NaN,NaN,NaN,NaN
4,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,NaN


In [11]:
df_equipment_only = df_downtime_repair_filter.merge(
    df_materials,
    how='left',
    left_on=['Оборудование', 'Дата'],
    right_on=['Заказ на ремонт.Транспортное средство (АПК)', 'Дата']
)
df_equipment_only

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость
0,A001,E001,2024-01-01,5,522,NaN,NaN,NaN,NaN
1,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0
2,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0
3,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN


In [12]:
df_equipment_only['источник'] = ''

In [13]:
df_equipment_only.loc[df_equipment_only['Ответственное подразделение'].notna(), 'источник'] = 'по_оборудованию'

In [14]:
df_equipment_only['источник'] = df_equipment_only['источник'].replace('', np.nan)

In [15]:
df_result_expanded

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине
3,A002,E002,2024-01-02,3,326,NaN,NaN,NaN,NaN,NaN
4,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,NaN


In [16]:
df_equipment_only

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,NaN,NaN,NaN,NaN,NaN
1,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0,по_оборудованию
2,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0,по_оборудованию
3,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,NaN


In [17]:
df_materials

,Заказ на ремонт.Транспортное средство (АПК),Дата,Ответственное подразделение,Номер_заказа,Стоимость
0,A001,2024-01-01,Отдел А,R-001,1000
1,A001,2024-01-01,Отдел Б,R-002,2000
2,A001,2024-01-01,Отдел В,R-003,3000
3,E002,2024-01-02,Отдел Г,R-004,4000
4,E002,2024-01-02,Отдел Д,R-005,5000
5,A003,2024-01-03,Отдел Е,R-006,6000


In [18]:
df_result = pd.concat([df_result_expanded, df_equipment_only], ignore_index=False)
df_result = df_result.dropna(subset='Ответственное подразделение')
df_result

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине
1,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0,по_оборудованию
2,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0,по_оборудованию


In [19]:
df_materials

,Заказ на ремонт.Транспортное средство (АПК),Дата,Ответственное подразделение,Номер_заказа,Стоимость
0,A001,2024-01-01,Отдел А,R-001,1000
1,A001,2024-01-01,Отдел Б,R-002,2000
2,A001,2024-01-01,Отдел В,R-003,3000
3,E002,2024-01-02,Отдел Г,R-004,4000
4,E002,2024-01-02,Отдел Д,R-005,5000
5,A003,2024-01-03,Отдел Е,R-006,6000


In [20]:
df_downtime_repair_filter

,Машина,Оборудование,Дата,Простой_часы,Простой_id
0,A001,E001,2024-01-01,5,522
1,A002,E002,2024-01-02,3,326
2,B40,E960,2024-01-10,10,101


In [21]:
df_result

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине
1,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0,по_оборудованию
2,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0,по_оборудованию


In [22]:
# Создаем временный столбец-индикатор
df_merge = df_downtime_repair_filter[['Машина', 'Оборудование', 'Дата', 'Простой_часы', 'Простой_id']].merge(
    df_result,
    on=['Машина', 'Оборудование', 'Дата', 'Простой_часы', 'Простой_id'],
    how='left',
    indicator=True
)
df_merge

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник,_merge
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине,both
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине,both
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине,both
3,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0,по_оборудованию,both
4,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0,по_оборудованию,both
5,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,NaN,left_only


In [23]:
# те простои, по которым нет ремонтов
df_downtime_without_materials = df_merge.query('_merge == "left_only"')
df_downtime_without_materials.loc[:, 'источник'] = 'нет_данных'
df_downtime_without_materials = df_downtime_without_materials.drop('_merge', axis=1)
df_downtime_without_materials

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
5,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,нет_данных


In [24]:
df_result_final = pd.concat([df_result, df_downtime_without_materials])
df_result_final

,Машина,Оборудование,Дата,Простой_часы,Простой_id,Заказ на ремонт.Транспортное средство (АПК),Ответственное подразделение,Номер_заказа,Стоимость,источник
0,A001,E001,2024-01-01,5,522,A001,Отдел А,R-001,1000.0,по_машине
1,A001,E001,2024-01-01,5,522,A001,Отдел Б,R-002,2000.0,по_машине
2,A001,E001,2024-01-01,5,522,A001,Отдел В,R-003,3000.0,по_машине
1,A002,E002,2024-01-02,3,326,E002,Отдел Г,R-004,4000.0,по_оборудованию
2,A002,E002,2024-01-02,3,326,E002,Отдел Д,R-005,5000.0,по_оборудованию
5,B40,E960,2024-01-10,10,101,NaN,NaN,NaN,NaN,нет_данных


In [ ]:
# число простоев, по которым нет работ
len(df_result_final.query('источник == "нет_данных"')['Простой_id'].unique())

1

In [31]:
# число уникальных простоев всего
len(df_result_final['Простой_id'].unique())

3

In [ ]:
# доля простоев без заведенных заказов на ремонт
len(df_result_final.query('источник == "нет_данных"')['Простой_id'].unique())/len(df_result_final['Простой_id'].unique())

0.3333333333333333